In [1]:
# Drive mount karo
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

# Data load karo
parsed_path = "/content/drive/MyDrive/ASAD_Thesis/processed_data/spark_logs_parsed.csv"

print("Loading data...")
df = pd.read_csv(parsed_path)
print(f"Shape: {df.shape}")
print(f"\nLabel Distribution:")
print(df['label'].value_counts())
print(f"\nColumns: {df.columns.tolist()}")
print("\nSample:")
print(df.head(3))

Mounted at /content/drive
Loading data...
Shape: (1661830, 6)

Label Distribution:
label
0    1659555
1       2033
2        242
Name: count, dtype: int64

Columns: ['app_id', 'container_id', 'raw_line', 'label', 'template', 'cluster_id']

Sample:
                           app_id                            container_id  \
0  application_1485248649253_0124  container_1485248649253_0124_02_000012   
1  application_1485248649253_0124  container_1485248649253_0124_02_000012   
2  application_1485248649253_0124  container_1485248649253_0124_02_000024   

                                            raw_line  label  \
0  17/06/08 14:57:53 INFO executor.Executor: Fini...      0   
1  17/06/08 14:58:03 INFO executor.Executor: Fini...      0   
2  17/06/08 15:00:56 INFO executor.Executor: Fini...      0   

                                            template  cluster_id  
0  17/06/08 14:57:53 INFO executor.Executor: Fini...           1  
1  17/06/08 <*> INFO executor.Executor: Finished ...     

In [2]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

print("Feature engineering shuru...")

# Label encode karo app_id aur container_id
le_app = LabelEncoder()
le_container = LabelEncoder()

df['app_id_enc'] = le_app.fit_transform(df['app_id'])
df['container_id_enc'] = le_container.fit_transform(df['container_id'])

# Binary label banao — Normal(0) vs Anomaly(1+2)
df['binary_label'] = df['label'].apply(lambda x: 0 if x == 0 else 1)

# Features select karo
features = ['cluster_id', 'app_id_enc', 'container_id_enc']
X = df[features]
y = df['binary_label']

print(f"Features: {features}")
print(f"X shape: {X.shape}")
print(f"\nBinary Label Distribution:")
print(y.value_counts())
print(f"\nAnomaly percentage: {(y==1).sum()/len(y)*100:.2f}%")

Feature engineering shuru...
Features: ['cluster_id', 'app_id_enc', 'container_id_enc']
X shape: (1661830, 3)

Binary Label Distribution:
binary_label
0    1659555
1       2275
Name: count, dtype: int64

Anomaly percentage: 0.14%


In [3]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

print("Train/Test split kar rahe hain...")

# Split pehle karo
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # class balance maintain karo
)

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"\nTrain anomalies: {(y_train==1).sum()}")
print(f"Test anomalies: {(y_test==1).sum()}")

# SMOTE sirf train pe apply karo
print("\nSMOTE apply kar rahe hain...")
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE:")
print(f"X_train shape: {X_train_sm.shape}")
print(f"Normal: {(y_train_sm==0).sum():,}")
print(f"Anomaly: {(y_train_sm==1).sum():,}")

Train/Test split kar rahe hain...
X_train: (1329464, 3)
X_test: (332366, 3)

Train anomalies: 1820
Test anomalies: 455

SMOTE apply kar rahe hain...

After SMOTE:
X_train shape: (2655288, 3)
Normal: 1,327,644
Anomaly: 1,327,644


In [4]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import numpy as np

print("Isolation Forest training shuru...")

# Isolation Forest unsupervised hai - SMOTE data nahi, original use karo
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.0014,  # 0.14% anomaly rate
    random_state=42,
    n_jobs=-1  # all CPU cores
)

iso_forest.fit(X_train)
print("Training complete!")

# Predict karo
print("Predicting...")
y_pred_iso = iso_forest.predict(X_test)

# Isolation Forest -1=anomaly, 1=normal — convert karo
y_pred_iso = np.where(y_pred_iso == -1, 1, 0)

print("\n=== Isolation Forest Results ===")
print(classification_report(y_test, y_pred_iso,
                            target_names=['Normal', 'Anomaly']))

# ROC-AUC
scores = iso_forest.decision_function(X_test)
roc_auc = roc_auc_score(y_test, -scores)
print(f"ROC-AUC Score: {roc_auc:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_iso)
print(f"\nConfusion Matrix:")
print(cm)

Isolation Forest training shuru...
Training complete!
Predicting...

=== Isolation Forest Results ===
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00    331911
     Anomaly       0.04      0.04      0.04       455

    accuracy                           1.00    332366
   macro avg       0.52      0.52      0.52    332366
weighted avg       1.00      1.00      1.00    332366

ROC-AUC Score: 0.9559

Confusion Matrix:
[[331483    428]
 [   437     18]]


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

print("Random Forest training shuru...")

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

# SMOTE wala balanced data use karo
rf.fit(X_train_sm, y_train_sm)
print("Training complete!")

# Predict
print("Predicting...")
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("\n=== Random Forest Results ===")
print(classification_report(y_test, y_pred_rf,
                            target_names=['Normal', 'Anomaly']))

roc_auc = roc_auc_score(y_test, y_prob_rf)
print(f"ROC-AUC Score: {roc_auc:.4f}")

cm = confusion_matrix(y_test, y_pred_rf)
print(f"\nConfusion Matrix:")
print(cm)

Random Forest training shuru...
Training complete!
Predicting...

=== Random Forest Results ===
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00    331911
     Anomaly       0.29      0.89      0.44       455

    accuracy                           1.00    332366
   macro avg       0.65      0.94      0.72    332366
weighted avg       1.00      1.00      1.00    332366

ROC-AUC Score: 0.9902

Confusion Matrix:
[[330940    971]
 [    52    403]]


In [2]:
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import numpy as np

print("One-Class SVM training shuru...")
print("(Sirf normal data pe train hoga - unsupervised)")

# Sirf normal training data use karo
X_train_normal = X_train[y_train == 0]
print(f"Training on {len(X_train_normal):,} normal samples...")

oc_svm = OneClassSVM(
    kernel='rbf',
    nu=0.0014,  # anomaly rate
    gamma='scale'
)

oc_svm.fit(X_train_normal)
print("Training complete!")

# Predict
print("Predicting...")
y_pred_svm = oc_svm.predict(X_test)
y_pred_svm = np.where(y_pred_svm == -1, 1, 0)

print("\n=== One-Class SVM Results ===")
print(classification_report(y_test, y_pred_svm,
                            target_names=['Normal', 'Anomaly']))

scores = oc_svm.decision_function(X_test)
roc_auc = roc_auc_score(y_test, -scores)
print(f"ROC-AUC Score: {roc_auc:.4f}")

cm = confusion_matrix(y_test, y_pred_svm)
print(f"\nConfusion Matrix:")
print(cm)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1. Data load ho raha hai...
Shape: (1661830, 6)
2. Feature engineering...
3. Train/test split...
4. SMOTE apply ho raha hai...

Sab ready hai!
X_train_sm: (2655288, 3)
X_test: (332366, 3)


In [3]:
# ============================================
# FULL PIPELINE - EK CELL MEIN
# ============================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Step 1: Load
print("1. Data load ho raha hai...")
parsed_path = "/content/drive/MyDrive/ASAD_Thesis/processed_data/spark_logs_parsed.csv"
df = pd.read_csv(parsed_path)
print(f"Shape: {df.shape}")

# Step 2: Features
print("2. Feature engineering...")
le_app = LabelEncoder()
le_container = LabelEncoder()
df['app_id_enc'] = le_app.fit_transform(df['app_id'])
df['container_id_enc'] = le_container.fit_transform(df['container_id'])
df['binary_label'] = df['label'].apply(lambda x: 0 if x == 0 else 1)

features = ['cluster_id', 'app_id_enc', 'container_id_enc']
X = df[features]
y = df['binary_label']

# Step 3: Split
print("3. Train/test split...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Step 4: SMOTE
print("4. SMOTE apply ho raha hai...")
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"\nSab ready hai!")
print(f"X_train_sm: {X_train_sm.shape}")
print(f"X_test: {X_test.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1. Data load ho raha hai...
Shape: (1661830, 6)
2. Feature engineering...
3. Train/test split...
4. SMOTE apply ho raha hai...

Sab ready hai!
X_train_sm: (2655288, 3)
X_test: (332366, 3)


In [4]:
from sklearn.linear_model import SGDOneClassSVM
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import numpy as np

results = {}

# ============================================
# Model 1: Isolation Forest
# ============================================
print("=" * 45)
print("1. Isolation Forest training...")
iso = IsolationForest(n_estimators=100, contamination=0.0014,
                      random_state=42, n_jobs=-1)
iso.fit(X_train)
y_pred_iso = np.where(iso.predict(X_test) == -1, 1, 0)
scores_iso = -iso.decision_function(X_test)
roc_iso = roc_auc_score(y_test, scores_iso)
results['Isolation Forest'] = {
    'roc_auc': roc_iso,
    'report': classification_report(y_test, y_pred_iso,
                target_names=['Normal', 'Anomaly'])
}
print(f"ROC-AUC: {roc_iso:.4f} ✅")

# ============================================
# Model 2: Random Forest
# ============================================
print("2. Random Forest training...")
rf = RandomForestClassifier(n_estimators=100, random_state=42,
                             n_jobs=-1, class_weight='balanced')
rf.fit(X_train_sm, y_train_sm)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]
roc_rf = roc_auc_score(y_test, y_prob_rf)
results['Random Forest'] = {
    'roc_auc': roc_rf,
    'report': classification_report(y_test, y_pred_rf,
                target_names=['Normal', 'Anomaly'])
}
print(f"ROC-AUC: {roc_rf:.4f} ✅")

# ============================================
# Model 3: SGD One-Class SVM
# ============================================
print("3. SGD One-Class SVM training...")
scaler = StandardScaler()
X_train_normal = X_train[y_train == 0]
X_train_scaled = scaler.fit_transform(X_train_normal)
X_test_scaled = scaler.transform(X_test)

oc_svm = SGDOneClassSVM(nu=0.0014, random_state=42)
oc_svm.fit(X_train_scaled)
y_pred_svm = np.where(oc_svm.predict(X_test_scaled) == -1, 1, 0)
scores_svm = -oc_svm.decision_function(X_test_scaled)
roc_svm = roc_auc_score(y_test, scores_svm)
results['One-Class SVM'] = {
    'roc_auc': roc_svm,
    'report': classification_report(y_test, y_pred_svm,
                target_names=['Normal', 'Anomaly'])
}
print(f"ROC-AUC: {roc_svm:.4f} ✅")

# ============================================
# Final Comparison
# ============================================
print("\n" + "=" * 45)
print("FINAL RESULTS COMPARISON")
print("=" * 45)
for model, res in results.items():
    print(f"\n{model} — ROC-AUC: {res['roc_auc']:.4f}")
    print(res['report'])

1. Isolation Forest training...
ROC-AUC: 0.9559 ✅
2. Random Forest training...
ROC-AUC: 0.9902 ✅
3. SGD One-Class SVM training...
ROC-AUC: 0.2388 ✅

FINAL RESULTS COMPARISON

Isolation Forest — ROC-AUC: 0.9559
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00    331911
     Anomaly       0.04      0.04      0.04       455

    accuracy                           1.00    332366
   macro avg       0.52      0.52      0.52    332366
weighted avg       1.00      1.00      1.00    332366


Random Forest — ROC-AUC: 0.9902
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00    331911
     Anomaly       0.29      0.89      0.44       455

    accuracy                           1.00    332366
   macro avg       0.65      0.94      0.72    332366
weighted avg       1.00      1.00      1.00    332366


One-Class SVM — ROC-AUC: 0.2388
              precision    recall  f1-score   support

      Normal   

In [5]:
import json
import os

save_dir = "/content/drive/MyDrive/ASAD_Thesis/results"
os.makedirs(save_dir, exist_ok=True)

# Results dictionary save karo
summary = {
    "Random Forest":     {"ROC_AUC": 0.9902, "F1_Anomaly": 0.44, "Recall_Anomaly": 0.89},
    "Isolation Forest":  {"ROC_AUC": 0.9559, "F1_Anomaly": 0.04, "Recall_Anomaly": 0.04},
    "One_Class_SVM":     {"ROC_AUC": 0.2388, "F1_Anomaly": 0.00, "Recall_Anomaly": 0.01}
}

with open(f"{save_dir}/traditional_ml_results.json", "w") as f:
    json.dump(summary, f, indent=4)

print("Results saved! ✅")
print(json.dumps(summary, indent=4))

Results saved! ✅
{
    "Random Forest": {
        "ROC_AUC": 0.9902,
        "F1_Anomaly": 0.44,
        "Recall_Anomaly": 0.89
    },
    "Isolation Forest": {
        "ROC_AUC": 0.9559,
        "F1_Anomaly": 0.04,
        "Recall_Anomaly": 0.04
    },
    "One_Class_SVM": {
        "ROC_AUC": 0.2388,
        "F1_Anomaly": 0.0,
        "Recall_Anomaly": 0.01
    }
}
